# Module 1 – Données : chargement et inspection

Charger les avis (`recommendations.csv`) et les jeux (`games.csv`) avec des types compacts, puis regarder : taille, types, valeurs manquantes, doublons.

On ne supprime et on n'impute **rien** à cette étape.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from config import *

NROWS = 5_000_000  # échantillon de développement (le fichier complet fait 41 M de lignes)

## 1. Avis

In [2]:
reviews = pd.read_csv(
    RAW_DIR / "recommendations.csv",
    usecols=["app_id", "date", "is_recommended", "hours", "helpful"],
    dtype={"app_id": "int32", "is_recommended": "bool", "hours": "float32", "helpful": "int32"},
    nrows=NROWS,
)
reviews["date"] = pd.to_datetime(reviews["date"], format="%Y-%m-%d", errors="coerce")

print("Shape :", reviews.shape)
reviews.dtypes

Shape : (5000000, 5)


app_id                     int32
helpful                    int32
date              datetime64[us]
is_recommended              bool
hours                    float32
dtype: object

In [3]:
print("% manquant par colonne :")
print(reviews.isna().mean() * 100)
print("Lignes en double :", reviews.duplicated().sum())

% manquant par colonne :
app_id            0.0
helpful           0.0
date              0.0
is_recommended    0.0
hours             0.0
dtype: float64


Lignes en double : 185657


Les « lignes en double » ne sont pas forcément de vrais doublons : deux joueurs peuvent laisser un avis sur le même jeu, le même jour, avec les mêmes heures de jeu. On ne les supprime donc pas.

### Mémoire

On mesure la mémoire de l'échantillon et on extrapole aux 41 M de lignes.

In [4]:
mem = reviews.memory_usage(deep=True).sum()
print(f"Échantillon : {mem / 1e6:.0f} Mo")
print(f"Fichier complet (estimation) : {mem / len(reviews) * 41_154_794 / 1e9:.1f} Go")

Échantillon : 105 Mo
Fichier complet (estimation) : 0.9 Go


Avec les types par défaut (`int64`, `float64`, et `date` en texte), le fichier complet prendrait plusieurs Go de plus. C'est surtout `date` qui pèse : en texte, chaque valeur est un objet Python d'environ 60 octets, contre 8 octets en `datetime64`.

**Avec moins de 12 Go de RAM**, on lit le fichier **par morceaux** et on ne garde que les lignes utiles. Par exemple, à l'étape suivante, seulement les avis des jeux tirés au sort :
```python
chunks = []
for chunk in pd.read_csv(path, usecols=..., dtype=..., chunksize=2_000_000):
    chunks.append(chunk[chunk["app_id"].isin(jeux_tires)])
reviews = pd.concat(chunks)
```

## 2. Jeux

In [5]:
games = pd.read_csv(RAW_DIR / "games.csv")
games["date_release"] = pd.to_datetime(games["date_release"], format="%Y-%m-%d", errors="coerce")

print("Shape :", games.shape)
games.dtypes

Shape : (50872, 13)


app_id                     int64
title                        str
date_release      datetime64[us]
win                         bool
mac                         bool
linux                       bool
rating                       str
positive_ratio             int64
user_reviews               int64
price_final              float64
price_original           float64
discount                 float64
steam_deck                  bool
dtype: object

In [6]:
print("% manquant par colonne :")
print(games.isna().mean() * 100)
print("app_id en double :", games["app_id"].duplicated().sum())

% manquant par colonne :
app_id            0.0
title             0.0
date_release      0.0
win               0.0
mac               0.0
linux             0.0
rating            0.0
positive_ratio    0.0
user_reviews      0.0
price_final       0.0
price_original    0.0
discount          0.0
steam_deck        0.0
dtype: float64
app_id en double : 0
